In [1]:
from pathlib import Path
import sys
import os
import numpy as np


try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive") # mount the drive
    DATA_PATH = Path("/content/drive/MyDrive") # the root of the drive
    DATA_ROOT = DATA_PATH / "cbc_pe_data" # the root of the data
else:
    PROJECT_ROOT = Path.cwd().parent
    sys.path.insert(0, str(PROJECT_ROOT))
    DATA_ROOT = PROJECT_ROOT / "data"


DATA_RAW = DATA_ROOT / "raw" # the root of the raw data
DATA_PROCESSED = DATA_ROOT / "processed" # the root of the processed data
MODELS_DIR = DATA_ROOT / "models" # the root of the models
RESULTS_DIR = DATA_ROOT / "results" # the root of the results
CHECKPOINTS_DIR = MODELS_DIR / "checkpoints" # the root of the checkpoints

for path in [DATA_RAW, DATA_PROCESSED, MODELS_DIR, RESULTS_DIR, CHECKPOINTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("DATA_RAW:", DATA_RAW)
print("MODELS_DIR:", MODELS_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("CHECKPOINTS_DIR:", CHECKPOINTS_DIR)



#Clone the repository if it doesn't exist
if IN_COLAB:
    REPO_ROOT = Path("/content/Gravitational-Waves-Lab")
    PROJECT_ROOT = REPO_ROOT / "cbc_pe"

    if not REPO_ROOT.exists():
        !git clone https://github.com/victorsh13/Gravitational-Waves-Lab.git /content/Gravitational-Waves-Lab

    print("REPO_ROOT exists:", REPO_ROOT.exists())
    print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())

    #Pull the latest version of the repository
    %cd /content/Gravitational-Waves-Lab
    !git pull
    !git status

    #Go to the project root
    %cd /content/Gravitational-Waves-Lab/cbc_pe
else:
    %cd /home/victor/gw/cbc_pe
    


# Add the project root to the python path
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("cwd:", Path.cwd())
print("sys.path[:3]:", sys.path[:3])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_ROOT: /content/drive/MyDrive/cbc_pe_data
DATA_RAW: /content/drive/MyDrive/cbc_pe_data/raw
MODELS_DIR: /content/drive/MyDrive/cbc_pe_data/models
RESULTS_DIR: /content/drive/MyDrive/cbc_pe_data/results
CHECKPOINTS_DIR: /content/drive/MyDrive/cbc_pe_data/models/checkpoints
REPO_ROOT exists: True
PROJECT_ROOT exists: True
/content/Gravitational-Waves-Lab
Already up to date.
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
/content/Gravitational-Waves-Lab/cbc_pe
cwd: /content/Gravitational-Waves-Lab/cbc_pe
sys.path[:3]: ['/content/Gravitational-Waves-Lab/cbc_pe', '/content', '/env/python']


In [2]:
from pathlib import Path
import numpy as np

from src.io import load_dataset_npz

dataset_id = "bbh_processed_4s_seobnrv4opt_snr10-25_n4500"

dataset_path = DATA_PROCESSED / f"{dataset_id}.npz"
split_path = DATA_PROCESSED / f"{dataset_id}_splits.npz"

batch = load_dataset_npz(dataset_path)

X = batch.X
y = batch.y
metadata = batch.metadata
network_snrs = np.array([ m["snr"]["final_network_snr"] for m in metadata], dtype=np.float32)
label_names = ["chirp_mass", "total_mass", "chi_eff"]


splits = np.load(split_path)

train_idx = splits["train_idx"]
val_idx = splits["val_idx"]
cal_idx = splits["cal_idx"]
test_idx = splits["test_idx"]

X_train, y_train_phys = X[train_idx], y[train_idx]
X_val, y_val_phys = X[val_idx], y[val_idx]
X_cal, y_cal_phys = X[cal_idx], y[cal_idx]
X_test, y_test_phys = X[test_idx], y[test_idx]

In [3]:
label_stats_path = DATA_PROCESSED / f"{dataset_id}_label_stats_train_only.npz"

y_params = np.load(label_stats_path)
y_mean = y_params["mean"]
y_std = y_params["std"]

y_train_std = (y_train_phys - y_mean) / y_std
y_val_std = (y_val_phys - y_mean) / y_std
y_cal_std = (y_cal_phys - y_mean) / y_std
y_test_std = (y_test_phys - y_mean) / y_std

## Datasets and Dataloaders

In [4]:
from torch.utils.data import DataLoader
from src.models.dataset import ArrayRegressionDataset

batch_size = 32

train_dataset = ArrayRegressionDataset(X_train, y_train_std)
val_dataset = ArrayRegressionDataset(X_val, y_val_std)
cal_dataset = ArrayRegressionDataset(X_cal, y_cal_std)
test_dataset = ArrayRegressionDataset(X_test, y_test_std)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
)

cal_loader = DataLoader(
    cal_dataset,
    batch_size=batch_size,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,

)

## Load the checkpoint

In [17]:
import torch
from src.models.network import SimpleCNN, SimpleCNN_v2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_id = "bbh_processed_4s_seobnrv4opt_snr10-25_n4500_SmoothL1Loss_emb64"

checkpoint_file_name = f"{checkpoint_id}_checkpoint.pt"
checkpoint_path = CHECKPOINTS_DIR / checkpoint_file_name

checkpoint = torch.load(checkpoint_path, map_location=device)

model_config = checkpoint["model_config"]

model = SimpleCNN(
    n_detectors=model_config["n_detectors"],
    n_outputs=model_config["n_outputs"],
    embedding_dim=model_config["embedding_dim"],
    dropout_conv=model_config["dropout_conv"],
    dropout_dense=model_config["dropout_dense"],
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded epoch:", checkpoint["epoch"])
print("Loaded best val loss:", checkpoint["best_val_loss"])

Loaded epoch: 147
Loaded best val loss: 0.20866821050643922


In [18]:
model

SimpleCNN(
  (block1): ConvBlock(
    (conv): Conv1d(3, 16, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 16, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout1d(p=0.05, inplace=False)
  )
  (block2): ConvBlock(
    (conv): Conv1d(16, 32, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 32, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout1d(p=0.05, inplace=False)
  )
  (block3): ConvBlock(
    (conv): Conv1d(32, 64, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 64, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout1d(p=0.05, inplace=False)
  )
  (block4): ConvBlock(
    (conv): Conv1d(64, 128, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 128, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): D

This function uses the model for the prediction inference

In [7]:
def predict_set(model, dataloader, device):
    model.eval()
    pred = []
    true = []
    emb = []

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)

            pred_batch, emb_batch = model(X_batch, return_embedding=True)

            pred.append(pred_batch.cpu().numpy())
            true.append(y_batch.numpy())
            emb.append(emb_batch.cpu().numpy())
            
    return np.concatenate(pred, axis=0), np.concatenate(true, axis=0), np.concatenate(emb, axis=0)

In [19]:
#Standardized predictions (y_std where used in the dataloader)
pred_train, y_train, emb_train = predict_set(model, train_loader, device)
pred_val, y_val, emb_val = predict_set(model, val_loader, device)
pred_cal, y_cal, emb_cal = predict_set(model, cal_loader, device)
pred_test, y_test, emb_test = predict_set(model, test_loader, device)




    """
    El coeficiente R2 mide qué fracción de la varianza del target explica el modelo.
        - R² = 1 -> predicción perfecta
        - R² = 0 -> igual que predecir siempre la media del target
        - R² < 0 -> peor que predecir siempre la media
    """
  

In [20]:
import pandas as pd
from src.models.evaluate import regression_metrics

metrics_train_std = regression_metrics(y_train, pred_train, label_names, "train_std")
metrics_val_std   = regression_metrics(y_val, pred_val, label_names, "val_std")
metrics_cal_std   = regression_metrics(y_cal, pred_cal, label_names, "cal_std")
metrics_test_std  = regression_metrics(y_test, pred_test, label_names, "test_std")

metrics_all_std = pd.concat(
    [metrics_train_std, metrics_val_std, metrics_cal_std, metrics_test_std],
    ignore_index=True
)

metrics_all_std

,split,label,MSE,RMSE,MAE,bias,median_abs_error,residual_std,R2
0,train_std,chirp_mass,0.101332,0.318327,0.247032,-0.011871,0.203837,0.318105,0.898668
1,train_std,total_mass,0.090925,0.301539,0.237791,-0.016710,0.198998,0.301075,0.909074
2,train_std,chi_eff,0.295469,0.543570,0.431441,-0.074780,0.363357,0.538402,0.704531
3,val_std,chirp_mass,0.384232,0.619865,0.471261,0.034104,0.363966,0.618926,0.620144
4,val_std,total_mass,0.337427,0.580885,0.466994,0.033144,0.400374,0.579938,0.666724
5,val_std,chi_eff,0.604841,0.777715,0.628376,-0.016455,0.507150,0.777541,0.372088
6,cal_std,chirp_mass,0.386602,0.621773,0.476406,0.012442,0.380397,0.621649,0.627394
7,cal_std,total_mass,0.337957,0.581340,0.467842,-0.012860,0.420087,0.581198,0.673599
8,cal_std,chi_eff,0.638824,0.799265,0.638257,0.020964,0.562995,0.798990,0.386517
9,test_std,chirp_mass,0.421866,0.649512,0.485763,0.004208,0.358557,0.649499,0.590968


In physical space

In [21]:
y_test_phys = y_test * y_std + y_mean
pred_test_phys = pred_test * y_std + y_mean

In [22]:
metrics_test_phys = regression_metrics(
    y_true=y_test_phys,
    y_pred=pred_test_phys,
    label_names=label_names,
    split_name="test_phys"
)

metrics_test_phys

,split,label,MSE,RMSE,MAE,bias,median_abs_error,residual_std,R2
0,test_phys,chirp_mass,112.346848,10.599380,7.927156,0.068671,5.851288,10.599161,0.590968
1,test_phys,total_mass,403.736298,20.093191,15.590345,0.335291,12.804665,20.090401,0.655563
2,test_phys,chi_eff,0.118044,0.343575,0.276852,-0.001011,0.247827,0.343573,0.400538


## 1. Compute some metrics

In [ ]:
from src.models.evaluate import inverse_standardize

pred_train_phys = inverse_standardize(pred_train, y_mean, y_std)
pred_val_phys   = inverse_standardize(pred_val,   y_mean, y_std)
pred_cal_phys   = inverse_standardize(pred_cal,   y_mean, y_std)
pred_test_phys  = inverse_standardize(pred_test,  y_mean, y_std)

y_train_phys = inverse_standardize(y_train, y_mean, y_std)
y_val_phys   = inverse_standardize(y_val,   y_mean, y_std)
y_cal_phys   = inverse_standardize(y_cal,   y_mean, y_std)
y_test_phys  = inverse_standardize(y_test,  y_mean, y_std)

In [ ]:
metrics_train_phys = regression_metrics(y_train_phys, pred_train_phys, label_names, "train_phys")
metrics_val_phys   = regression_metrics(y_val_phys,   pred_val_phys,   label_names, "val_phys")
metrics_cal_phys   = regression_metrics(y_cal_phys,   pred_cal_phys,   label_names, "cal_phys")
metrics_test_phys  = regression_metrics(y_test_phys,  pred_test_phys,  label_names, "test_phys")

metrics_all_phys = pd.concat(
    [metrics_train_phys, metrics_val_phys, metrics_cal_phys, metrics_test_phys],
    ignore_index=True
)

metrics_all_phys

## 2. Plot pred vs true

Qué mirar:

- Si los puntos siguen la diagonal.
- Si hay saturación en masas altas.
- Si hay regresión a la media.
- Si chi_eff está comprimido cerca de cero.

Mi predicción: chi_eff tendrá bastante regresión a la media.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_pred_vs_true(y_true, y_pred, label_names, split_name="test"):
    for j, label in enumerate(label_names):
        true = y_true[:, j]
        pred = y_pred[:, j]

        min_val = min(true.min(), pred.min())
        max_val = max(true.max(), pred.max())

        plt.figure(figsize=(8, 8))
        plt.scatter(true, pred, s=12, alpha=0.6)
        plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
        plt.xlabel(f"True {label}")
        plt.ylabel(f"Predicted {label}")
        plt.title(f"{split_name}: predicted vs true — {label}")
        plt.grid(True, alpha=0.3)
        plt.show()

In [ ]:
plot_pred_vs_true(
    y_true=y_test_phys,
    y_pred=pred_test_phys,
    label_names=label_names,
    split_name="test_phys"
)

## 3. Residuals per label

In [ ]:
def plot_residuals(y_true, y_pred, label_names, split_name="test"):
    residual = y_pred - y_true

    for j, label in enumerate(label_names):
        plt.figure(figsize=(6, 4))
        plt.hist(residual[:, j], bins=40, alpha=0.8)
        plt.axvline(0.0, linestyle="--")
        plt.xlabel(f"Residual: pred - true ({label})")
        plt.ylabel("Count")
        plt.title(f"{split_name}: residual distribution — {label}")
        plt.grid(True, alpha=0.3)
        plt.show()

In [ ]:
plot_residuals(y_test_phys, pred_test_phys, label_names, "test_phys")

In [ ]:
def plot_residual_vs_true(y_true, y_pred, label_names, split_name="test"):
    residual = y_pred - y_true

    for j, label in enumerate(label_names):
        plt.figure(figsize=(6, 4))
        plt.scatter(y_true[:, j], residual[:, j], s=12, alpha=0.6)
        plt.axhline(0.0, linestyle="--")
        plt.xlabel(f"True {label}")
        plt.ylabel(f"Residual: pred - true")
        plt.title(f"{split_name}: residual vs true — {label}")
        plt.grid(True, alpha=0.3)
        plt.show()

Esto es más importante que el histograma. Te dirá dónde falla el modelo.

In [ ]:
plot_residual_vs_true(y_test_phys, pred_test_phys, label_names, "test_phys")

## 4. Absolute error vs true value

Este plot es clave para Mondrian. Si el error aumenta con masa o depende de chi_eff, entonces tiene sentido usar bins condicionados.

In [ ]:
def plot_abs_error_vs_true(y_true, y_pred, label_names, split_name="test"):
    abs_error = np.abs(y_pred - y_true)

    for j, label in enumerate(label_names):
        plt.figure(figsize=(6, 4))
        plt.scatter(y_true[:, j], abs_error[:, j], s=12, alpha=0.6)
        plt.xlabel(f"True {label}")
        plt.ylabel(f"Absolute error")
        plt.title(f"{split_name}: absolute error vs true — {label}")
        plt.grid(True, alpha=0.3)
        plt.show()

In [ ]:
plot_abs_error_vs_true(y_test_phys, pred_test_phys, label_names, "test_phys")

## 5. Error vs SNR

In [ ]:
snr_test = network_snrs[test_idx]

In [ ]:
def plot_abs_error_vs_quantity(quantity, y_true, y_pred, label_names, quantity_name, split_name="test"):
    abs_error = np.abs(y_pred - y_true)

    for j, label in enumerate(label_names):
        plt.figure(figsize=(6, 4))
        plt.scatter(quantity, abs_error[:, j], s=12, alpha=0.6)
        plt.xlabel(quantity_name)
        plt.ylabel(f"Absolute error in {label}")
        plt.title(f"{split_name}: abs error vs {quantity_name} — {label}")
        plt.grid(True, alpha=0.3)
        plt.show()

In [ ]:
plot_abs_error_vs_quantity(
    quantity=snr_test,
    y_true=y_test_phys,
    y_pred=pred_test_phys,
    label_names=label_names,
    quantity_name="network SNR",
    split_name="test_phys"
)

For bins

In [ ]:
def metrics_by_quantity_bins(quantity, y_true, y_pred, label_names, bin_edges, quantity_name):
    rows = []

    for b in range(len(bin_edges) - 1):
        lo = bin_edges[b]
        hi = bin_edges[b + 1]

        mask = (quantity >= lo) & (quantity < hi)

        if mask.sum() == 0:
            continue

        residual = y_pred[mask] - y_true[mask]
        abs_error = np.abs(residual)

        for j, label in enumerate(label_names):
            rows.append({
                "quantity": quantity_name,
                "bin": f"[{lo:.2f}, {hi:.2f})",
                "count": int(mask.sum()),
                "label": label,
                "MAE": abs_error[:, j].mean(),
                "RMSE": np.sqrt((residual[:, j] ** 2).mean()),
                "bias": residual[:, j].mean(),
                "median_abs_error": np.median(abs_error[:, j]),
            })

    return pd.DataFrame(rows)

In [ ]:


snr_edges = np.linspace(10, 25, 7)

metrics_snr_test = metrics_by_quantity_bins(
    quantity=snr_test,
    y_true=y_test_phys,
    y_pred=pred_test_phys,
    label_names=label_names,
    bin_edges=snr_edges,
    quantity_name="network_snr"
)


snr_by_label = metrics_snr_test.sort_values(by="label", kind="stable")


display(snr_by_label)

## 6. Save predictions and embeddings

In [ ]:
output_path = RESULTS_DIR / f"{checkpoint_id}_predictions_embeddings.npz"

np.savez(
    output_path,

    pred_train=pred_train,
    pred_val=pred_val,
    pred_cal=pred_cal,
    pred_test=pred_test,

    y_train=y_train,
    y_val=y_val,
    y_cal=y_cal,
    y_test=y_test,

    pred_train_phys=pred_train_phys,
    pred_val_phys=pred_val_phys,
    pred_cal_phys=pred_cal_phys,
    pred_test_phys=pred_test_phys,

    y_train_phys=y_train_phys,
    y_val_phys=y_val_phys,
    y_cal_phys=y_cal_phys,
    y_test_phys=y_test_phys,

    emb_train=emb_train,
    emb_val=emb_val,
    emb_cal=emb_cal,
    emb_test=emb_test,

    idx_train=train_idx,
    idx_val=val_idx,
    idx_cal=cal_idx,
    idx_test=test_idx,

    y_mean=y_mean,
    y_std=y_std,

    label_names=np.array(label_names),

    best_epoch=checkpoint["epoch"],
    best_val_loss=checkpoint["best_val_loss"],
    checkpoint_path=str(checkpoint_path),
)

print("Saved:", output_path)